[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aims-foundations/torch_measure/blob/main/tutorials/doubly_robust_sparse_benchmarks.ipynb)

# Doubly Robust Model for Sparse Benchmark Matrices

AI benchmark leaderboards are rarely complete: not every model is evaluated on every task.
When this missingness is informative — e.g. frontier models are evaluated only on hard
benchmarks, or cheap models skip expensive evaluations — a naive IRT fit is biased.

The **DoublyRobustModel** corrects for this by learning a residual correction on top of
a base IRT model, trained with inverse-propensity-weighted (IPW) loss:

    final_prediction(i, j) = base_model(i, j) + correction(i, j)

The propensity weighting during training ensures the correction compensates for
the observation bias, making predictions consistent even under MNAR.

This tutorial demonstrates:
1. How informative missingness biases a naive Rasch fit
2. How `DoublyRobustModel` learns a bias-correcting residual
3. Comparing dense predictions between the base and DR model

## 1. Setup

In [ ]:
try:
    import google.colab
    !git clone https://github.com/aims-foundations/torch_measure.git
    !pip install -e "torch_measure[test]" -q
except ImportError:
    pass

import sys, types
sys.modules.setdefault('sentence_transformers',
                       types.ModuleType('sentence_transformers'))
if not hasattr(sys.modules['sentence_transformers'], 'SentenceTransformer'):
    sys.modules['sentence_transformers'].SentenceTransformer = object

import numpy as np
import torch
import matplotlib.pyplot as plt

from torch_measure.models import Rasch, DoublyRobustModel
from torch_measure.models._predictor import predict_dense

plt.rcParams["figure.dpi"] = 110
torch.manual_seed(42)
np.random.seed(42)

print("Setup complete.")

## 2. Generate Sparse Benchmark Data

We simulate a **complete** Rasch response matrix, then apply MNAR missingness:
high-ability models are less likely to be observed on easy items (mimicking the
pattern where frontier models skip trivial benchmarks).

In [ ]:
n_models, n_items = 60, 80

ability = torch.randn(n_models)
difficulty = torch.randn(n_items)

logits = ability.unsqueeze(1) - difficulty.unsqueeze(0)
probs = torch.sigmoid(logits)
data_full = torch.bernoulli(probs)

# MNAR missingness
obs_logits = -0.6 * ability.unsqueeze(1) + 0.6 * difficulty.unsqueeze(0)
obs_probs = torch.sigmoid(obs_logits).clamp(0.15, 0.95)
obs_mask = torch.bernoulli(obs_probs).bool()

for i in range(n_models):
    if not obs_mask[i].any():
        obs_mask[i, torch.randint(n_items, (1,))] = True

data_sparse = data_full.clone()
data_sparse[~obs_mask] = float("nan")

obs_rate = obs_mask.float().mean().item()
print(f"Response matrix: {n_models} models x {n_items} items")
print(f"Observation rate: {obs_rate:.1%}")
print(f"True mean accuracy: {data_full.mean():.3f}")

## 3. Fit a Base Rasch Model

First we fit a standard Rasch model on the sparse data. Because the missingness
is informative, this fit will be biased — ability estimates for high-ability models
will be pulled down (they're mostly observed on hard items).

In [ ]:
base_model = Rasch(n_subjects=n_models, n_items=n_items)
history = base_model.fit(data_sparse, max_epochs=300, verbose=False)

print(f"Base model final loss: {history['losses'][-1]:.4f}")

# Evaluate: how well does the base model recover the full matrix?
base_preds = predict_dense(base_model).detach()
base_mse = ((base_preds - data_full) ** 2).mean().item()
print(f"Base model MSE on full matrix: {base_mse:.4f}")

## 4. Fit the Doubly Robust Correction

Now we wrap the base model in `DoublyRobustModel` and fit the residual correction.
The correction is trained with IPW-weighted loss — each observation is weighted by
`1/propensity`, so the model learns to correct for the selection bias.

In [ ]:
dr_model = DoublyRobustModel(base_model, clip_propensity=(0.05, 0.95))
dr_history = dr_model.fit(data_sparse, max_epochs=300, verbose=False)

print(f"DR model final loss: {dr_history['losses'][-1]:.4f}")

dr_preds = predict_dense(dr_model).detach()
dr_mse = ((dr_preds - data_full) ** 2).mean().item()
print(f"DR model MSE on full matrix: {dr_mse:.4f}")
print(f"Improvement over base: {(base_mse - dr_mse) / base_mse:.1%}")

## 5. Compare Ability Estimates

The base model's ability estimates are biased because of MNAR. Let's see how the
DR model's effective abilities (base + correction) compare to the true values.

In [ ]:
# Per-model mean prediction as a proxy for "effective ability"
base_mean_pred = base_preds.mean(dim=1).numpy()
dr_mean_pred = dr_preds.mean(dim=1).numpy()
true_mean = data_full.mean(dim=1).numpy()

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

axes[0].scatter(true_mean, base_mean_pred, alpha=0.6, s=20)
axes[0].plot([0, 1], [0, 1], "k--", lw=1)
axes[0].set_xlabel("True mean accuracy")
axes[0].set_ylabel("Predicted mean accuracy")
axes[0].set_title("Base Rasch Model")

axes[1].scatter(true_mean, dr_mean_pred, alpha=0.6, s=20, color="tab:orange")
axes[1].plot([0, 1], [0, 1], "k--", lw=1)
axes[1].set_xlabel("True mean accuracy")
axes[1].set_ylabel("Predicted mean accuracy")
axes[1].set_title("DoublyRobustModel")

plt.tight_layout()
plt.show()

base_bias = np.abs(base_mean_pred - true_mean).mean()
dr_bias = np.abs(dr_mean_pred - true_mean).mean()
print(f"Mean absolute bias — Base: {base_bias:.4f}, DR: {dr_bias:.4f}")

## 6. Correction Magnitude

How large is the learned correction? We expect it to be small (the base model
already captures most of the signal) but systematic (correcting the MNAR bias).

In [ ]:
correction = (dr_preds - base_preds).numpy()

print(f"Correction stats:")
print(f"  Mean:   {correction.mean():.4f}")
print(f"  Std:    {correction.std():.4f}")
print(f"  Range:  [{correction.min():.4f}, {correction.max():.4f}]")

plt.figure(figsize=(6, 3.5))
plt.hist(correction.ravel(), bins=50, edgecolor="white", alpha=0.7)
plt.axvline(0, color="k", linestyle="--", lw=1)
plt.xlabel("Correction (DR pred - Base pred)")
plt.ylabel("Count")
plt.title("Distribution of DR Correction Term")
plt.tight_layout()
plt.show()

## 7. Summary

Key findings:

- A standard Rasch model fitted on MNAR-sparse data gives **biased** predictions
  and ability estimates.
- `DoublyRobustModel` learns a residual correction layer on top of the frozen base,
  trained with IPW-weighted loss to correct for the observation bias.
- The correction is typically small in magnitude but reduces prediction error on
  the full (unobserved) matrix.

Typical workflow for sparse leaderboard evaluation:
```python
from torch_measure.models import Rasch, DoublyRobustModel

base = Rasch(n_subjects=..., n_items=...)
base.fit(sparse_matrix, method="mle")

dr = DoublyRobustModel(base)
dr.fit(sparse_matrix, max_epochs=300)

# Dense predictions corrected for observation bias
predictions = predict_dense(dr)
```

**References:**
- Robins, Rotnitzky & Zhao (1994). Estimation of regression coefficients when some
  regressors are not always observed. *JASA*.
- `torch_measure.models.DoublyRobustModel` API documentation